In [ ]:
pip install --upgrade "nudenet>=3.4.2";

## Code

### get_files
"""
Returns a list of image file paths in a directory.

Args:
    folder_path (str): Path to the directory to search for images. Default: "files".
    extensions (tuple): Allowed image file extensions. Default: (".jpeg", ".jpg", ".png").

Returns:
    List[str]: List of image file paths found in the directory.
"""

In [ ]:
from pathlib import Path

folder = Path("files")
extensions = (".jpeg", ".jpg", ".png")

files = [
    str(file) 
    for file in folder.iterdir() 
        if 
            file.is_file() and 
            file.suffix.lower() in extensions
]

### detect_nude

Performs batch detection of nudity and explicit content in a list of images using NudeNet.  

In [ ]:
from nudenet import NudeDetector

valid_labels = [
    "FEMALE_GENITALIA_COVERED", "FEMALE_GENITALIA_EXPOSED",
    "MALE_GENITALIA_COVERED", "MALE_GENITALIA_EXPOSED",
    "ANUS_EXPOSED", "ANUS_COVERED",
]
detector = NudeDetector()
images = detector.detect_batch(files)

nude_results = [
    {
        "name": Path(file).name,
        "path": file,
        "detect_data": [
            detect for detect in image 
                if detect['class'] in valid_labels
        ]
    }
    for file, image in zip(files, images)
]

nude_results

### 

In [ ]:
import cv2

images = [
    {
        "name": file['name'],
        "path": file['path'],
        "image": {
            "original": cv2.imread(file['path']) 
        },
        "detect_data": file['detect_data']
    } for file in nude_results
]

images

### censor_image

Applies pixelation censorship to detected regions in an image based on detection data.

In [ ]:
first = images[0]

detect_data = first['detect_data']
image = first["image"]["original"].copy()
output_path = "output"
pixel_size = 3

for detect in detect_data:
    x, y, w, h = detect['box']
    roi_small = cv2.resize(
        image[y:y+h, x:x+w], 
        (w // pixel_size, h // pixel_size), 
        interpolation=cv2.INTER_LINEAR
    )
    roi_pixelated = cv2.resize(roi_small, (w, h), interpolation=cv2.INTER_NEAREST)
    image[y:y+h, x:x+w] = roi_pixelated

image
# cv2.imwrite(f"{output_path}/{first["name"]}", image)

### add_box_with_padding

Calculates a new bounding box for censorship by adding padding to the detected region.

In [ ]:
box = detect_data[0]['box'] # x, y, w, h = detect_data[0]['box']
pixel_size = pixel_size
padding = 10

x, y, w, h = [
    box[0]+padding, 
    box[1]+padding, 
    box[2]-2*padding, 
    box[3]-2*padding
]

[x, y, w, h]

### add_censor

Applies pixelation censorship to a region in an image based on detection data.

In [ ]:
roi_small = cv2.resize(
    image[y:y+h, x:x+w], 
    (w // pixel_size, h // pixel_size), 
    interpolation=cv2.INTER_LINEAR
)
roi_pixelated = cv2.resize(
    roi_small, 
    (w, h), 
    interpolation=cv2.INTER_NEAREST
)
image[y:y+h, x:x+w] = roi_pixelated

### add_debug_square

In [ ]:
square_image = first["image"]["original"].copy()

square_color = (255, 255, 255)
square_thickness = 2
square_x, square_y, square_w, square_h = images[0]['detect_data'][2]['box']


cv2.rectangle(
    square_image, 
    (square_x, square_y, square_w, square_h), 
    square_color, 
    2
)

# cv2.imwrite(f"debug/12.png", square_image)

### add_debug_info


In [ ]:
line_color = (255, 255, 255) 
padding_color = (255, 0, 0) 

debug_detect_data = images[0]['detect_data'][2]
debug_image = first["image"]["original"].copy()
debug_name = first["name"]
debug_output_path = "debug"

debug_class = debug_detect_data["class"]
debug_box = debug_detect_data['box']
debug_score = debug_detect_data["score"]

cv2.rectangle(
    debug_image, 
    debug_box, 
    line_color, 
    2
)
cv2.putText(
    debug_image, 
    debug_class,       
    (
        debug_box[0], 
        debug_box[1] - 28
    ), 
    cv2.FONT_HERSHEY_SIMPLEX,
    0.64,
    (255, 255, 255),
    2
)
cv2.putText(
    debug_image, 
    str(debug_score),       
    (
        debug_box[0], 
        debug_box[1] - 8
    ), 
    cv2.FONT_HERSHEY_SIMPLEX,
    0.48,
    (255, 255, 255),
    2
)

padding_line = (
    debug_box[0]+padding, 
    debug_box[1]+padding, 
    debug_box[2]-2*padding, 
    debug_box[3]-2*padding
)
cv2.rectangle(
    debug_image, 
    padding_line, 
    padding_color, 
    1
)

cv2.imwrite(f"{debug_output_path}/{debug_name}", debug_image)